In [4]:
import os
print(os.getcwd())
os.chdir(r'C:\Users\awet0')

C:\Users\awet0\OneDrive\ACIT\Master Thesis\JupytherLab\cloudsim-custom-implementation\analysis\Experiment 1


In [49]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
from collections import defaultdict
import seaborn as sns

# === Font settings for EPS export with readable text ===
mpl.rcParams.update({
    'ps.useafm': True,
    'pdf.use14corefonts': True,
    'text.usetex': False,
    'font.size': 13
})

# === Map for power model aliases ===
power_model_aliases = {
    "105.0,112.0,118.0": "MG3",
    "86.0,89.4,92.6": "MG4",
    "93.7,97.0,101.0": "MG5"
}

def map_power_model(model_str):
    for key, alias in power_model_aliases.items():
        if model_str.startswith(key):
            return alias
    return model_str

# === Reusable function to extract metrics from a log file ===
def extract_resource_usage(csv_path):
    df = pd.read_csv(csv_path, delimiter=";")

    dc_names = []
    pe_usage_pct, mips_usage_pct, ram_usage_pct = [], [], []
    bw_usage_pct, storage_usage_pct = [], []

    for dc_name, group in df.groupby('datacenter_name'):
        dc_name_map = {
            "Datacenter_1": "DC1",
            "Datacenter_2": "DC2",
            "Datacenter_3": "DC3",
        }
        short_name = dc_name_map.get(dc_name, dc_name)  
        dc_names.append(short_name)

        pe_total = group['number_of_pes'].sum()
        pe_available = group['available_pes'].sum()
        pe_usage_pct.append((pe_total - pe_available) / pe_total * 100 if pe_total else 0)

        mips_total = group['mips'].sum()
        mips_available = group['available_mips'].sum()
        mips_usage_pct.append((mips_total - mips_available) / mips_total * 100 if mips_total else 0)

        ram_total = group['ram'].sum()
        ram_available = group['available_ram'].sum()
        ram_usage_pct.append((ram_total - ram_available) / ram_total * 100 if ram_total else 0)

        bw_total = group['bw'].sum()
        bw_available = group['available_bw'].sum()
        bw_usage_pct.append((bw_total - bw_available) / bw_total * 100 if bw_total else 0)

        storage_total = group['storage'].sum()
        storage_available = group['available_storage'].sum()
        storage_usage_pct.append((storage_total - storage_available) / storage_total * 100 if storage_total else 0)

    return dc_names, [pe_usage_pct, mips_usage_pct, ram_usage_pct, bw_usage_pct, storage_usage_pct]


def extract_host_activity(csv_path):
    df = pd.read_csv(csv_path, delimiter=";")
    dc_names, total_hosts_list, powered_on_hosts_list, active_on_hosts_list = [], [], [], []

    for dc_name, group in df.groupby('datacenter_name'):
        dc_name_map = {
            "Datacenter_1": "DC1",
            "Datacenter_2": "DC2",
            "Datacenter_3": "DC3",
        }
        short_name = dc_name_map.get(dc_name, dc_name)  
        dc_names.append(short_name)

        total_hosts = group['host_id'].nunique()
        powered_on = group[group['power_on'] == True]['host_id'].nunique()
        active = group[group['active'] == True]['host_id'].nunique()

        total_hosts_list.append(total_hosts)
        powered_on_hosts_list.append(powered_on)
        active_on_hosts_list.append(active)

    return dc_names, total_hosts_list, powered_on_hosts_list, active_on_hosts_list



power_model_aliases = {
    "105.0,112.0,118.0": "MG3",
    "86.0,89.4,92.6": "MG4",
    "93.7,97.0,101.0": "MG5",
    #"0.0,92.5,110.0": "CelsiusV80"
}

dc_name_map = {
    "Datacenter_1": "DC1",
    "Datacenter_2": "DC2",
    "Datacenter_3": "DC3",
}

def map_power_model(model_str):
    for key, alias in power_model_aliases.items():
        if str(model_str).startswith(key):
            return alias
    return model_str
def apply_dc_mapping(df):
    return df.assign(datacenter_name=df['datacenter_name'].map(dc_name_map))


def extract_utilization_summaries(csv_path):
    df = pd.read_csv(csv_path, delimiter=';')

    # Map power model
    df['power_model'] = df['power_model'].apply(map_power_model)

    # Map datacenter name
    if 'datacenter_name' in df.columns:
        df['datacenter_name'] = df['datacenter_name'].map(dc_name_map)

    # Compute utilization metrics
    df['ram_utilization'] = (df['ram'] - df['available_ram']) / df['ram']
    df['bw_utilization'] = (df['bw'] - df['available_bw']) / df['bw']
    df['storage_utilization'] = (df['storage'] - df['available_storage']) / df['storage']
    df['disk_io_utilization'] = df['disk_I/O'] / 1024  # KB to MB or MB to GB depending

    # Helper to aggregate summaries
    def summarize(col):
        return df.groupby(['datacenter_name', 'power_model'])[col].agg(
            ['mean', 'std', 'min', 'max', 'count']).reset_index()

    cpu_summary     = summarize('cpu_utilization')
    ram_summary     = summarize('ram_utilization')
    bw_summary      = summarize('bw_utilization')
    storage_summary = summarize('storage_utilization')
    io_summary      = summarize('disk_io_utilization')

    return cpu_summary, ram_summary, bw_summary, storage_summary, io_summary



exp1_path = "OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/logs/exp1/BLdata.csv"
exp2_path = "OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/logs/exp2/BLdata.csv"

# === Load and process both experiments ===
dc1_names, values1 = extract_resource_usage(exp1_path)
dc2_names, values2 = extract_resource_usage(exp2_path)

dc1_names, total1, powered1, active1 = extract_host_activity(exp1_path)
dc2_names, total2, powered2, active2 = extract_host_activity(exp2_path)


cpu_dist_summary1, ram_dist_summary1, bw_dist_summary1, storage_dist_summary1, disk_io_dist_summary1 = extract_utilization_summaries(exp1_path)
cpu_dist_summary2, ram_dist_summary2, bw_dist_summary2, storage_dist_summary2, disk_io_dist_summary2 = extract_utilization_summaries(exp2_path)


# === Prepare subplot layout ===
bar_width = 0.12
resources = ["PEs", "MIPS", "RAM", "BW", "Storage"]
index1 = np.arange(len(dc1_names))
index2 = np.arange(len(dc2_names))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

# === Plotting function for each subplot ===
def plot_resource_bar(ax, dc_names, index, values, label):
    for i, (res, data) in enumerate(zip(resources, values)):
        ax.bar(index + i * bar_width, data, bar_width, label=res)
    ax.set_xlabel("Datacenters")
    ax.set_xticks(index + bar_width * len(resources) / 2)
    ax.set_xticklabels(dc_names, rotation=30)
    ax.set_ylim(0, 105)
    ax.text(0.5, 1.05, label, transform=ax.transAxes,
        fontsize=16, fontweight='bold', va='bottom', ha='center')
    ax.legend(fontsize=10, loc='upper right', facecolor='white', frameon=False, bbox_to_anchor=(0.4, 1))


# === Subplot A and B ===
plot_resource_bar(axes[0], dc1_names, index1, values1, "Experiment 1")
plot_resource_bar(axes[1], dc2_names, index2, values2, "Experiment 2")
axes[0].set_ylabel("Resource utilization (%)", fontweight='bold')
axes[1].set_ylabel("Resource utilization (%)", fontweight='bold')
plt.tight_layout()


# === Save as EPS ===
plt.savefig("C:/Users/awet0/OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/analysis/Experiment 1/figures/Baseline/EX1_EX2_resource_utilization_subplot.eps", format='eps', dpi=300)
plt.close()






bar_width = 0.2
resources = ["Total hosts", "Powered on hosts", "Active hosts"]
index1 = np.arange(len(dc1_names))
index2 = np.arange(len(dc2_names))

values1 = [total1, powered1, active1]
values2 = [total2, powered2, active2]

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=False)

# === Plot function ===
def plot_host_activity(ax, index, dc_names, values, label):
    for i, (rname, data) in enumerate(zip(resources, values)):
        ax.bar(index + i * bar_width, data, bar_width, label=rname)
    ax.set_xlabel("Datacenters", fontsize=12)
    ax.set_xticks(index + bar_width * len(resources) / 2)
    ax.set_xticklabels(dc_names, rotation=30, fontsize=10)
    ax.text(0.5, 1.05, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', ha='center', va='bottom')
    ax.legend(fontsize=10, loc='upper right', frameon=False, facecolor='white')

# === Plot both subplots ===
plot_host_activity(axes[0], index1, dc1_names, values1, "Experiment 1")
plot_host_activity(axes[1], index2, dc2_names, values2, "Experiment 2")

axes[0].set_ylabel("Number of Hosts", fontsize=12)
axes[1].set_ylabel("Number of Hosts", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 1])

# === Save ===
#plt.savefig("host_activity_comparison.pdf", format='pdf', dpi=300)
plt.savefig("C:/Users/awet0/OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/analysis/Experiment 1/figures/Baseline/host_activity_comparison.eps", format='eps', dpi=300)
plt.close()



def get_heatmap_data(cpu, ram, bw, storage, io, selected_cols):
    return [
        cpu.pivot(index="datacenter_name", columns="power_model", values="mean")[selected_cols],
        ram.pivot(index="datacenter_name", columns="power_model", values="mean")[selected_cols],
        bw.pivot(index="datacenter_name", columns="power_model", values="mean")[selected_cols],
        storage.pivot(index="datacenter_name", columns="power_model", values="mean")[selected_cols],
        io.pivot(index="datacenter_name", columns="power_model", values="mean")[selected_cols],
    ]



#columns_to_plot = ["CelsiusV80", "MG3", "MG4", "MG5"]
columns_to_plot = ["MG3", "MG4", "MG5"]
titles = ["CPU", "RAM", "Bandwidth", "Storage", "Disk I/O"]



exp1_data = get_heatmap_data(cpu_dist_summary1, ram_dist_summary1, bw_dist_summary1, storage_dist_summary1, disk_io_dist_summary1, columns_to_plot)
exp2_data = get_heatmap_data(cpu_dist_summary2, ram_dist_summary2, bw_dist_summary2, storage_dist_summary2, disk_io_dist_summary2, columns_to_plot)


fig, axs = plt.subplots(5, 2, figsize=(18, 20))
#cbar_ax1 = fig.add_axes([0.91, 0.55, 0.015, 0.35])  # Right colorbar for Exp2
#cbar_ax2 = fig.add_axes([0.91, 0.1, 0.015, 0.35])   # Right colorbar for Exp1

heatmap_kwargs = {
    "annot": True,
    "fmt": ".2f",
    "cmap": "YlGnBu",
    "linewidths": 0.5,
    "annot_kws": {"fontsize": 20, "fontweight": "bold"},
    
    
}

font_kwargs = {
    "fontsize": 17,
    "fontweight": "bold"
}

# Plot
for i in range(5):
    for j, data in enumerate([exp1_data, exp2_data]):
        ax = axs[i, j]
        pivot = data[i]
        pivot = pivot[[col for col in columns_to_plot if col in pivot.columns]]
        
        heat = sns.heatmap(pivot, ax=ax, **heatmap_kwargs)
        #ax.set_title(f"{titles[i]} Utilization - {'Exp1' if j == 0 else 'Exp2'}", **font_kwargs)
        ax.set_xlabel("Server Type", fontsize=15)
        if j == 0:
            ax.set_ylabel("Datacenter", fontsize=17)
        else:
            ax.set_ylabel("")

        # Tick styling
        ax.tick_params(labelsize=15)
        #for label in ax.get_xticklabels() + ax.get_yticklabels():
            #label.set_fontweight("bold")


fig.text(0.25, 0.96, "Experiment 1", fontsize=16, fontweight="bold", ha="center")
fig.text(0.75, 0.96, "Experiment 2", fontsize=16, fontweight="bold", ha="center")

# Final layout and save
plt.tight_layout(rect=[0, 0, 1, 0.95])   
plt.savefig("C:/Users/awet0/OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/analysis/Experiment 1/figures/Baseline/combined_experiment_heatmaps.pdf", format='pdf', dpi=300)
plt.savefig("C:/Users/awet0/OneDrive/ACIT/Master Thesis/JupytherLab/cloudsim-custom-implementation/analysis/Experiment 1/figures/Baseline/combined_experiment_heatmaps.eps", format='eps', dpi=300)
plt.close()